## [控制流](https://docs.langchain.com/oss/python/langgraph/graph-api-control-flow)

### 分支结构 - 静态

#### 一、并行分支

In [ ]:
import operator
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class State(TypedDict):
    topic: str
    poems: Annotated[list, operator.add]

def a(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的五言绝句")
    return {"poems": [poem.content]}

def b(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的七言绝句")
    return {"poems": [poem.content]}


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_edge(START, "a")
builder.add_edge(START, "b")
builder.add_edge("a", END)
builder.add_edge("b", END)
graph = builder.compile()
result = graph.invoke({"topic": "猫", "poems": []})
print(result)

In [ ]:
from IPython.display import display

display(graph)

#### 二、条件分支

In [ ]:
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class State(TypedDict):
    topic: str
    type: str
    poem: str

def a(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的五言绝句")
    return {"poem": poem.content}

def b(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的七言绝句")
    return {"poem": poem.content}

def conditional_edge(state: State) -> Literal["a", "b"]:
    if "五言绝句" == state["type"]:
        return "a"
    else:
        return "b"


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_conditional_edges(START, conditional_edge)
builder.add_edge("a", END)
builder.add_edge("b", END)
graph = builder.compile()
result = graph.invoke({"topic": "猫", "type": "五言绝句"})
print(result)

##### 2.2 路由到多节点

In [ ]:
import operator
from typing import TypedDict, Literal, Annotated, Sequence
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class State(TypedDict):
    topic: str
    type: str
    poems: Annotated[list, operator.add]

def a(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的五言绝句")
    return {"poems": [poem.content]}

def b(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的七言绝句")
    return {"poems": [poem.content]}

def c(state: State):
    poem = model.invoke(f"请写一首短篇关于{state["topic"]}的词")
    return {"poems": [poem.content]}

def conditional_edge(state: State) -> Sequence[Literal["5_poem", "7_poem", "ci_poem"]]:
    if "诗" == state["type"]:
        return ["5_poem", "7_poem"]
    else:
        return ["ci_poem"]


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_node(c)
# 这里添加 path_map 是为了之后生成正确的图结构
path_map = {"5_poem": "a", "7_poem": "b", "ci_poem": "c"}
builder.add_conditional_edges(START, conditional_edge, path_map=path_map)
builder.add_edge("a", END)
builder.add_edge("b", END)
builder.add_edge("c", END)
graph = builder.compile()
result = graph.invoke({"topic": "猫", "type": "词"})
print(result)

In [ ]:
from IPython.display import display

display(graph)

##### 2.3 延迟节点

在常规流程运行结束后，再在额外的超步中触发执行

In [ ]:
import operator
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class State(TypedDict):
    topic: str
    poems: Annotated[list, operator.add]

def a(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的五言绝句")
    return {"poems": [poem.content]}

def b(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的七言绝句")
    return {"poems": [poem.content]}

def check(state: State):
    if len(state["poems"]) == 2:
        print("检查通过")
    else:
        print("检查失败")


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_node(check, defer=True)
builder.add_edge(START, "a")
builder.add_edge(START, "b")
builder.add_edge(START, "check")
builder.add_edge("a", END)
builder.add_edge("b", END)
builder.add_edge("check", END)
graph = builder.compile()
result = graph.invoke({"topic": "猫", "poems": []})
print(result)